In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ✅ VANILLA LDM TRAINING CLASS-LEAFSPOT

import torch, os
from torch import nn, optim
from torchvision import transforms as T
from torchvision.utils import save_image
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from torch.amp import autocast, GradScaler
import torch.nn.functional as F

# ================= CONFIG =================
IMG_SIZE = 224
LATENT_DIM = 1024   # you can reduce to 512 if needed
BATCH_SIZE = 8
EPOCHS = 500
LR = 2e-4
TIMESTEPS = 1000
DATA_PATH = "/content/drive/MyDrive/BambooLeaf/LeafSpot"

SAVE_PATH = "/content/drive/MyDrive/BambooLeaf/LDMLS_224"
os.makedirs(SAVE_PATH, exist_ok=True)

# ================= DEVICE =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = GradScaler()

# ================= DATASET =================
class SingleClassImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.paths = [os.path.join(root_dir, f) for f in os.listdir(root_dir)
                      if f.endswith((".png", ".jpg", ".jpeg"))]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, 0

# ================= TRANSFORM =================
transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

dataset = SingleClassImageDataset(DATA_PATH, transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# ================= MODELS =================

# ✅ ENCODER (224 → 7)
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.ReLU(),       # 224 → 112
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(), # 112 → 56
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(), # 56 → 28
            nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(), # 28 → 14
            nn.Conv2d(512, 512, 4, 2, 1), nn.ReLU(),                      # 14 → 7
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, latent_dim)
        )

    def forward(self, x):
        return self.net(x)

# ✅ DECODER (7 → 224)
class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 512 * 7 * 7)

        self.net = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(), # 7 → 14
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(), # 14 → 28
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(), # 28 → 56
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),   # 56 → 112
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh()                          # 112 → 224
        )

    def forward(self, z):
        x = self.fc(z).view(-1, 512, 7, 7)
        return self.net(x)

# ✅ LATENT UNET (Diffusion in latent space)
class LatentUNet(nn.Module):
    def __init__(self, latent_dim, timesteps):
        super().__init__()
        self.time_embed = nn.Embedding(timesteps, latent_dim)

        self.net = nn.Sequential(
            nn.Linear(latent_dim, 1024), nn.ReLU(),
            nn.Linear(1024, 1024), nn.ReLU(),
            nn.Linear(1024, latent_dim)
        )

    def forward(self, x, t):
        return self.net(x + self.time_embed(t))

# ================= INIT =================
encoder = Encoder(LATENT_DIM).to(device)
decoder = Decoder(LATENT_DIM).to(device)
unet = LatentUNet(LATENT_DIM, TIMESTEPS).to(device)

optimizer = optim.Adam(
    list(encoder.parameters()) +
    list(decoder.parameters()) +
    list(unet.parameters()),
    lr=LR
)

# ================= DIFFUSION SCHEDULER =================
betas = torch.linspace(1e-4, 0.02, TIMESTEPS).to(device)
alphas = 1. - betas
alpha_hat = torch.cumprod(alphas, dim=0)

# ================= KL LOSS =================
def kl_divergence(mu):
    return -0.5 * torch.sum(1 + 0 - mu.pow(2) - 1, dim=1).mean()

# ================= TRAIN =================
print("🚀 Training Started (224x224)...")

for epoch in range(EPOCHS):
    encoder.train()
    decoder.train()
    unet.train()

    pbar = tqdm(dataloader)

    for images, _ in pbar:
        images = images.to(device)

        with autocast(device_type=device.type):

            # ===== AUTOENCODER =====
            z = encoder(images)
            recon = decoder(z)

            recon_loss = F.mse_loss(recon, images)
            kl_loss = kl_divergence(z)
            ae_loss = recon_loss + 0.001 * kl_loss

            # ===== DIFFUSION =====
            t = torch.randint(0, TIMESTEPS, (z.size(0),), device=device).long()
            noise = torch.randn_like(z)

            a_hat = alpha_hat[t].unsqueeze(1)
            noisy_z = torch.sqrt(a_hat) * z + torch.sqrt(1 - a_hat) * noise

            pred_noise = unet(noisy_z, t)
            diffusion_loss = F.mse_loss(pred_noise, noise)

            # ===== TOTAL =====
            loss = ae_loss + diffusion_loss

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        pbar.set_description(f"Epoch {epoch+1} Loss: {loss.item():.4f}")

    # ===== SAVE =====
    if (epoch + 1) % 20 == 0:
        with torch.no_grad():
            sample_images, _ = next(iter(dataloader))
            z_sample = encoder(sample_images.to(device))
            samples = decoder(z_sample)

            save_image((samples + 1) / 2,
                       f"{SAVE_PATH}/sample_epoch{epoch+1}.png",
                       nrow=4)

            torch.save(encoder.state_dict(), f"{SAVE_PATH}/encoder_{epoch+1}.pth")
            torch.save(decoder.state_dict(), f"{SAVE_PATH}/decoder_{epoch+1}.pth")
            torch.save(unet.state_dict(),    f"{SAVE_PATH}/unet_{epoch+1}.pth")

            print(f"✅ Saved epoch {epoch+1}")

print("🎉 Training Completed!")

🚀 Training Started (224x224)...


Epoch 20 Loss: 0.8966: 100%|██████████| 161/161 [00:04<00:00, 38.16it/s]


✅ Saved epoch 20


Epoch 40 Loss: 0.7828: 100%|██████████| 161/161 [00:04<00:00, 39.38it/s]


✅ Saved epoch 40


...

✅ Saved epoch 460


Epoch 480 Loss: 0.6656: 100%|██████████| 161/161 [00:04<00:00, 39.32it/s]


✅ Saved epoch 480


Epoch 500 Loss: 0.6704: 100%|██████████| 161/161 [00:04<00:00, 39.73it/s]


✅ Saved epoch 500
🎉 Training Completed!


In [ ]:
#CODE TO DELETE FOLDER

from google.colab import drive
import shutil

# Delete folder
folder_path = '/content/drive/MyDrive/BambooLeaf/LDMLS_224/LSGenerated'

shutil.rmtree(folder_path)

print("Folder deleted successfully!")

Folder deleted successfully!


In [ ]:

#VANILLA LDM GENERATION CLASS-LEAFSPOT
import torch, os
from torchvision.utils import save_image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import torchvision.transforms as T
from itertools import cycle

# ================= CONFIG =================
LATENT_DIM = 1024
IMG_SIZE = 224
DATA_PATH = "/content/drive/MyDrive/BambooLeaf/LeafSpot"
OUTPUT_DIR = "/content/drive/MyDrive/BambooLeaf/LDMLS_224/LSGenerated"
TOTAL_IMAGES = 5000

os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= DATASET =================
class SingleClassImageDataset(Dataset):
    def __init__(self, root_dir):
        self.paths = [os.path.join(root_dir, f) for f in os.listdir(root_dir)
                      if f.endswith((".png", ".jpg", ".jpeg"))]
        self.transform = T.Compose([
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img)

loader = DataLoader(SingleClassImageDataset(DATA_PATH),
                    batch_size=8, shuffle=True)

# ================= MODELS (224 VERSION) =================
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.ReLU(),       # 224 → 112
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(), # 112 → 56
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(), # 56 → 28
            nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(), # 28 → 14
            nn.Conv2d(512, 512, 4, 2, 1), nn.ReLU(),                      # 14 → 7
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, latent_dim)
        )
    def forward(self, x): return self.net(x)


class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 512 * 7 * 7)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(), # 7 → 14
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(), # 14 → 28
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(), # 28 → 56
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),   # 56 → 112
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh()                          # 112 → 224
        )
    def forward(self, z):
        return self.net(self.fc(z).view(-1, 512, 7, 7))

# ================= LOAD MODELS =================
encoder = Encoder(LATENT_DIM).to(device)
decoder = Decoder(LATENT_DIM).to(device)

encoder.load_state_dict(torch.load("/content/drive/MyDrive/BambooLeaf/LDMLS_224/encoder_500.pth"))
decoder.load_state_dict(torch.load("/content/drive/MyDrive/BambooLeaf/LDMLS_224/decoder_500.pth"))

encoder.eval()
decoder.eval()

print("✅ Models loaded successfully (224x224)")

# ================= GENERATION =================
@torch.no_grad()
def generate():
    count = 0
    loop_loader = cycle(loader)

    while count < TOTAL_IMAGES:
        images = next(loop_loader).to(device)

        z = encoder(images)
        imgs = (decoder(z) + 1) / 2

        for j, img in enumerate(imgs):
            if count + j >= TOTAL_IMAGES:
                break
            save_image(img, f"{OUTPUT_DIR}/GEN_{count + j + 1:05}.png")

        count += len(imgs)

    print(f"✅ Generated {TOTAL_IMAGES} images")

# ================= RUN =================
generate()

✅ Models loaded successfully (224x224)
✅ Generated 5000 images


In [ ]:
import os
import zipfile

# ====== CONFIG ======
folder_path = "/content/drive/MyDrive/BambooLeaf/LDMHL_224/HLGenerated"   # your generated images folder
zip_path = folder_path + ".zip"                  # output zip file
# =====================

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, folder_path)
            zipf.write(file_path, arcname)

print(f"✅ ZIP file created successfully at: {zip_path}")

✅ ZIP file created successfully at: /content/drive/MyDrive/BambooLeaf/LDMHL_224/HLGenerated.zip


In [ ]:
# ✅ VANILLA LDM TRAINING CLASS-HEALTHY

import torch, os
from torch import nn, optim
from torchvision import transforms as T
from torchvision.utils import save_image
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from torch.amp import autocast, GradScaler
import torch.nn.functional as F

# ================= CONFIG =================
IMG_SIZE = 224
LATENT_DIM = 1024   # you can reduce to 512 if needed
BATCH_SIZE = 8
EPOCHS = 500
LR = 2e-4
TIMESTEPS = 1000
DATA_PATH = "/content/drive/MyDrive/BambooLeaf/Healthy"

SAVE_PATH = "/content/drive/MyDrive/BambooLeaf/LDMHL_224"
os.makedirs(SAVE_PATH, exist_ok=True)

# ================= DEVICE =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = GradScaler()

# ================= DATASET =================
class SingleClassImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.paths = [os.path.join(root_dir, f) for f in os.listdir(root_dir)
                      if f.endswith((".png", ".jpg", ".jpeg"))]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, 0

# ================= TRANSFORM =================
transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

dataset = SingleClassImageDataset(DATA_PATH, transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# ================= MODELS =================

# ✅ ENCODER (224 → 7)
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.ReLU(),       # 224 → 112
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(), # 112 → 56
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(), # 56 → 28
            nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(), # 28 → 14
            nn.Conv2d(512, 512, 4, 2, 1), nn.ReLU(),                      # 14 → 7
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, latent_dim)
        )

    def forward(self, x):
        return self.net(x)

# ✅ DECODER (7 → 224)
class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 512 * 7 * 7)

        self.net = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(), # 7 → 14
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(), # 14 → 28
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(), # 28 → 56
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),   # 56 → 112
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh()                          # 112 → 224
        )

    def forward(self, z):
        x = self.fc(z).view(-1, 512, 7, 7)
        return self.net(x)

# ✅ LATENT UNET (Diffusion in latent space)
class LatentUNet(nn.Module):
    def __init__(self, latent_dim, timesteps):
        super().__init__()
        self.time_embed = nn.Embedding(timesteps, latent_dim)

        self.net = nn.Sequential(
            nn.Linear(latent_dim, 1024), nn.ReLU(),
            nn.Linear(1024, 1024), nn.ReLU(),
            nn.Linear(1024, latent_dim)
        )

    def forward(self, x, t):
        return self.net(x + self.time_embed(t))

# ================= INIT =================
encoder = Encoder(LATENT_DIM).to(device)
decoder = Decoder(LATENT_DIM).to(device)
unet = LatentUNet(LATENT_DIM, TIMESTEPS).to(device)

optimizer = optim.Adam(
    list(encoder.parameters()) +
    list(decoder.parameters()) +
    list(unet.parameters()),
    lr=LR
)

# ================= DIFFUSION SCHEDULER =================
betas = torch.linspace(1e-4, 0.02, TIMESTEPS).to(device)
alphas = 1. - betas
alpha_hat = torch.cumprod(alphas, dim=0)

# ================= KL LOSS =================
def kl_divergence(mu):
    return -0.5 * torch.sum(1 + 0 - mu.pow(2) - 1, dim=1).mean()

# ================= TRAIN =================
print("🚀 Training Started (224x224)...")

for epoch in range(EPOCHS):
    encoder.train()
    decoder.train()
    unet.train()

    pbar = tqdm(dataloader)

    for images, _ in pbar:
        images = images.to(device)

        with autocast(device_type=device.type):

            # ===== AUTOENCODER =====
            z = encoder(images)
            recon = decoder(z)

            recon_loss = F.mse_loss(recon, images)
            kl_loss = kl_divergence(z)
            ae_loss = recon_loss + 0.001 * kl_loss

            # ===== DIFFUSION =====
            t = torch.randint(0, TIMESTEPS, (z.size(0),), device=device).long()
            noise = torch.randn_like(z)

            a_hat = alpha_hat[t].unsqueeze(1)
            noisy_z = torch.sqrt(a_hat) * z + torch.sqrt(1 - a_hat) * noise

            pred_noise = unet(noisy_z, t)
            diffusion_loss = F.mse_loss(pred_noise, noise)

            # ===== TOTAL =====
            loss = ae_loss + diffusion_loss

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        pbar.set_description(f"Epoch {epoch+1} Loss: {loss.item():.4f}")

    # ===== SAVE =====
    if (epoch + 1) % 20 == 0:
        with torch.no_grad():
            sample_images, _ = next(iter(dataloader))
            z_sample = encoder(sample_images.to(device))
            samples = decoder(z_sample)

            save_image((samples + 1) / 2,
                       f"{SAVE_PATH}/sample_epoch{epoch+1}.png",
                       nrow=4)

            torch.save(encoder.state_dict(), f"{SAVE_PATH}/encoder_{epoch+1}.pth")
            torch.save(decoder.state_dict(), f"{SAVE_PATH}/decoder_{epoch+1}.pth")
            torch.save(unet.state_dict(),    f"{SAVE_PATH}/unet_{epoch+1}.pth")

            print(f"✅ Saved epoch {epoch+1}")

print("🎉 Training Completed!")

🚀 Training Started (224x224)...


Epoch 20 Loss: 0.8949: 100%|██████████| 198/198 [00:05<00:00, 39.02it/s]


✅ Saved epoch 20


Epoch 40 Loss: 0.8499: 100%|██████████| 198/198 [00:04<00:00, 40.95it/s]


✅ Saved epoch 40


...

✅ Saved epoch 460


Epoch 480 Loss: 0.5623: 100%|██████████| 198/198 [00:05<00:00, 39.16it/s]


✅ Saved epoch 480


Epoch 500 Loss: 0.5287: 100%|██████████| 198/198 [00:05<00:00, 39.23it/s]


✅ Saved epoch 500
🎉 Training Completed!


In [ ]:
#VANILLA LDM GENERATION CLASS-HEALTHY
import torch, os
from torchvision.utils import save_image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import torchvision.transforms as T
from itertools import cycle

# ================= CONFIG =================
LATENT_DIM = 1024
IMG_SIZE = 224
DATA_PATH = "/content/drive/MyDrive/BambooLeaf/Healthy"
OUTPUT_DIR = "/content/drive/MyDrive/BambooLeaf/LDMHL_224/HLGenerated"
TOTAL_IMAGES = 5000

os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= DATASET =================
class SingleClassImageDataset(Dataset):
    def __init__(self, root_dir):
        self.paths = [os.path.join(root_dir, f) for f in os.listdir(root_dir)
                      if f.endswith((".png", ".jpg", ".jpeg"))]
        self.transform = T.Compose([
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img)

loader = DataLoader(SingleClassImageDataset(DATA_PATH),
                    batch_size=8, shuffle=True)

# ================= MODELS (224 VERSION) =================
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.ReLU(),       # 224 → 112
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(), # 112 → 56
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(), # 56 → 28
            nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(), # 28 → 14
            nn.Conv2d(512, 512, 4, 2, 1), nn.ReLU(),                      # 14 → 7
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, latent_dim)
        )
    def forward(self, x): return self.net(x)


class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 512 * 7 * 7)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(), # 7 → 14
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(), # 14 → 28
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(), # 28 → 56
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),   # 56 → 112
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh()                          # 112 → 224
        )
    def forward(self, z):
        return self.net(self.fc(z).view(-1, 512, 7, 7))

# ================= LOAD MODELS =================
encoder = Encoder(LATENT_DIM).to(device)
decoder = Decoder(LATENT_DIM).to(device)

encoder.load_state_dict(torch.load("/content/drive/MyDrive/BambooLeaf/LDMHL_224/encoder_500.pth"))
decoder.load_state_dict(torch.load("/content/drive/MyDrive/BambooLeaf/LDMHL_224/decoder_500.pth"))

encoder.eval()
decoder.eval()

print("✅ Models loaded successfully (224x224)")

# ================= GENERATION =================
@torch.no_grad()
def generate():
    count = 0
    loop_loader = cycle(loader)

    while count < TOTAL_IMAGES:
        images = next(loop_loader).to(device)

        z = encoder(images)
        imgs = (decoder(z) + 1) / 2

        for j, img in enumerate(imgs):
            if count + j >= TOTAL_IMAGES:
                break
            save_image(img, f"{OUTPUT_DIR}/GEN_{count + j + 1:05}.png")

        count += len(imgs)

    print(f"✅ Generated {TOTAL_IMAGES} images")

# ================= RUN =================
generate()

✅ Models loaded successfully (224x224)
✅ Generated 5000 images


In [ ]:
# ✅ VANILLA LDM TRAINING CLASS-INSECTDAMAGE

import torch, os
from torch import nn, optim
from torchvision import transforms as T
from torchvision.utils import save_image
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from torch.amp import autocast, GradScaler
import torch.nn.functional as F

# ================= CONFIG =================
IMG_SIZE = 224
LATENT_DIM = 1024   # you can reduce to 512 if needed
BATCH_SIZE = 8
EPOCHS = 500
LR = 2e-4
TIMESTEPS = 1000
DATA_PATH = "/content/drive/MyDrive/BambooLeaf/InsectDamage"

SAVE_PATH = "/content/drive/MyDrive/BambooLeaf/LDMID_224"
os.makedirs(SAVE_PATH, exist_ok=True)

# ================= DEVICE =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = GradScaler()

# ================= DATASET =================
class SingleClassImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.paths = [os.path.join(root_dir, f) for f in os.listdir(root_dir)
                      if f.endswith((".png", ".jpg", ".jpeg"))]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, 0

# ================= TRANSFORM =================
transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

dataset = SingleClassImageDataset(DATA_PATH, transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# ================= MODELS =================

# ✅ ENCODER (224 → 7)
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.ReLU(),       # 224 → 112
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(), # 112 → 56
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(), # 56 → 28
            nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(), # 28 → 14
            nn.Conv2d(512, 512, 4, 2, 1), nn.ReLU(),                      # 14 → 7
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, latent_dim)
        )

    def forward(self, x):
        return self.net(x)

# ✅ DECODER (7 → 224)
class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 512 * 7 * 7)

        self.net = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(), # 7 → 14
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(), # 14 → 28
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(), # 28 → 56
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),   # 56 → 112
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh()                          # 112 → 224
        )

    def forward(self, z):
        x = self.fc(z).view(-1, 512, 7, 7)
        return self.net(x)

# ✅ LATENT UNET (Diffusion in latent space)
class LatentUNet(nn.Module):
    def __init__(self, latent_dim, timesteps):
        super().__init__()
        self.time_embed = nn.Embedding(timesteps, latent_dim)

        self.net = nn.Sequential(
            nn.Linear(latent_dim, 1024), nn.ReLU(),
            nn.Linear(1024, 1024), nn.ReLU(),
            nn.Linear(1024, latent_dim)
        )

    def forward(self, x, t):
        return self.net(x + self.time_embed(t))

# ================= INIT =================
encoder = Encoder(LATENT_DIM).to(device)
decoder = Decoder(LATENT_DIM).to(device)
unet = LatentUNet(LATENT_DIM, TIMESTEPS).to(device)

optimizer = optim.Adam(
    list(encoder.parameters()) +
    list(decoder.parameters()) +
    list(unet.parameters()),
    lr=LR
)

# ================= DIFFUSION SCHEDULER =================
betas = torch.linspace(1e-4, 0.02, TIMESTEPS).to(device)
alphas = 1. - betas
alpha_hat = torch.cumprod(alphas, dim=0)

# ================= KL LOSS =================
def kl_divergence(mu):
    return -0.5 * torch.sum(1 + 0 - mu.pow(2) - 1, dim=1).mean()

# ================= TRAIN =================
print("🚀 Training Started (224x224)...")

for epoch in range(EPOCHS):
    encoder.train()
    decoder.train()
    unet.train()

    pbar = tqdm(dataloader)

    for images, _ in pbar:
        images = images.to(device)

        with autocast(device_type=device.type):

            # ===== AUTOENCODER =====
            z = encoder(images)
            recon = decoder(z)

            recon_loss = F.mse_loss(recon, images)
            kl_loss = kl_divergence(z)
            ae_loss = recon_loss + 0.001 * kl_loss

            # ===== DIFFUSION =====
            t = torch.randint(0, TIMESTEPS, (z.size(0),), device=device).long()
            noise = torch.randn_like(z)

            a_hat = alpha_hat[t].unsqueeze(1)
            noisy_z = torch.sqrt(a_hat) * z + torch.sqrt(1 - a_hat) * noise

            pred_noise = unet(noisy_z, t)
            diffusion_loss = F.mse_loss(pred_noise, noise)

            # ===== TOTAL =====
            loss = ae_loss + diffusion_loss

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        pbar.set_description(f"Epoch {epoch+1} Loss: {loss.item():.4f}")

    # ===== SAVE =====
    if (epoch + 1) % 20 == 0:
        with torch.no_grad():
            sample_images, _ = next(iter(dataloader))
            z_sample = encoder(sample_images.to(device))
            samples = decoder(z_sample)

            save_image((samples + 1) / 2,
                       f"{SAVE_PATH}/sample_epoch{epoch+1}.png",
                       nrow=4)

            torch.save(encoder.state_dict(), f"{SAVE_PATH}/encoder_{epoch+1}.pth")
            torch.save(decoder.state_dict(), f"{SAVE_PATH}/decoder_{epoch+1}.pth")
            torch.save(unet.state_dict(),    f"{SAVE_PATH}/unet_{epoch+1}.pth")

            print(f"✅ Saved epoch {epoch+1}")

print("🎉 Training Completed!")

🚀 Training Started (224x224)...


Epoch 20 Loss: 0.9140: 100%|██████████| 155/155 [00:04<00:00, 38.23it/s]


✅ Saved epoch 20


Epoch 40 Loss: 0.7797: 100%|██████████| 155/155 [00:03<00:00, 38.95it/s]


✅ Saved epoch 40


...

✅ Saved epoch 460


Epoch 480 Loss: 0.5455: 100%|██████████| 155/155 [00:04<00:00, 38.22it/s]


✅ Saved epoch 480


Epoch 500 Loss: 0.5988: 100%|██████████| 155/155 [00:04<00:00, 38.53it/s]


✅ Saved epoch 500
🎉 Training Completed!


In [ ]:
import torch, os, shutil
from torchvision.utils import save_image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import torchvision.transforms as T
from itertools import cycle

# ================= CONFIG =================
LATENT_DIM = 1024
IMG_SIZE = 224
DATA_PATH = "/content/drive/MyDrive/BambooLeaf/InsectDamage"
OUTPUT_DIR = "/content/drive/MyDrive/BambooLeaf/LDMID_224/IDGenerated"
ZIP_PATH = "/content/drive/MyDrive/BambooLeaf/LDMID_224/IDGenerated.zip"
TOTAL_IMAGES = 5000

os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= DATASET =================
class SingleClassImageDataset(Dataset):
    def __init__(self, root_dir):
        self.paths = [os.path.join(root_dir, f) for f in os.listdir(root_dir)
                      if f.endswith((".png", ".jpg", ".jpeg"))]
        self.transform = T.Compose([
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img)

loader = DataLoader(SingleClassImageDataset(DATA_PATH),
                    batch_size=8, shuffle=True)

# ================= MODELS =================
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.Conv2d(512, 512, 4, 2, 1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, latent_dim)
        )
    def forward(self, x): return self.net(x)

class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 512 * 7 * 7)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh()
        )
    def forward(self, z):
        return self.net(self.fc(z).view(-1, 512, 7, 7))

# ================= LOAD =================
encoder = Encoder(LATENT_DIM).to(device)
decoder = Decoder(LATENT_DIM).to(device)

encoder.load_state_dict(torch.load("/content/drive/MyDrive/BambooLeaf/LDMID_224/encoder_500.pth"))
decoder.load_state_dict(torch.load("/content/drive/MyDrive/BambooLeaf/LDMID_224/decoder_500.pth"))

encoder.eval()
decoder.eval()

print("✅ Models loaded successfully (224x224)")

# ================= GENERATION =================
@torch.no_grad()
def generate():
    count = 0
    loop_loader = cycle(loader)

    while count < TOTAL_IMAGES:
        images = next(loop_loader).to(device)

        z = encoder(images)
        imgs = (decoder(z) + 1) / 2

        for j, img in enumerate(imgs):
            if count + j >= TOTAL_IMAGES:
                break
            save_image(img, f"{OUTPUT_DIR}/GEN_{count + j + 1:05}.png")

        count += len(imgs)

    print(f"✅ Generated {TOTAL_IMAGES} images")

# ================= ZIP FUNCTION =================
def create_zip():
    print("📦 Creating ZIP file...")
    shutil.make_archive(ZIP_PATH.replace(".zip", ""), 'zip', OUTPUT_DIR)
    print(f"✅ ZIP saved at: {ZIP_PATH}")

# ================= RUN =================
generate()
create_zip()

✅ Models loaded successfully (224x224)
✅ Generated 5000 images
📦 Creating ZIP file...
✅ ZIP saved at: /content/drive/MyDrive/BambooLeaf/LDMID_224/IDGenerated.zip


In [ ]:
# ✅ VANILLA LDM TRAINING CLASS-LEAFBLIGHT

import torch, os
from torch import nn, optim
from torchvision import transforms as T
from torchvision.utils import save_image
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from torch.amp import autocast, GradScaler
import torch.nn.functional as F

# ================= CONFIG =================
IMG_SIZE = 224
LATENT_DIM = 1024   # you can reduce to 512 if needed
BATCH_SIZE = 8
EPOCHS = 500
LR = 2e-4
TIMESTEPS = 1000
DATA_PATH = "/content/drive/MyDrive/BambooLeaf/LeafBlight"

SAVE_PATH = "/content/drive/MyDrive/BambooLeaf/LDMLB_224"
os.makedirs(SAVE_PATH, exist_ok=True)

# ================= DEVICE =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = GradScaler()

# ================= DATASET =================
class SingleClassImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.paths = [os.path.join(root_dir, f) for f in os.listdir(root_dir)
                      if f.endswith((".png", ".jpg", ".jpeg"))]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, 0

# ================= TRANSFORM =================
transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

dataset = SingleClassImageDataset(DATA_PATH, transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# ================= MODELS =================

# ✅ ENCODER (224 → 7)
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.ReLU(),       # 224 → 112
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(), # 112 → 56
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(), # 56 → 28
            nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(), # 28 → 14
            nn.Conv2d(512, 512, 4, 2, 1), nn.ReLU(),                      # 14 → 7
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, latent_dim)
        )

    def forward(self, x):
        return self.net(x)

# ✅ DECODER (7 → 224)
class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 512 * 7 * 7)

        self.net = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(), # 7 → 14
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(), # 14 → 28
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(), # 28 → 56
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),   # 56 → 112
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh()                          # 112 → 224
        )

    def forward(self, z):
        x = self.fc(z).view(-1, 512, 7, 7)
        return self.net(x)

# ✅ LATENT UNET (Diffusion in latent space)
class LatentUNet(nn.Module):
    def __init__(self, latent_dim, timesteps):
        super().__init__()
        self.time_embed = nn.Embedding(timesteps, latent_dim)

        self.net = nn.Sequential(
            nn.Linear(latent_dim, 1024), nn.ReLU(),
            nn.Linear(1024, 1024), nn.ReLU(),
            nn.Linear(1024, latent_dim)
        )

    def forward(self, x, t):
        return self.net(x + self.time_embed(t))

# ================= INIT =================
encoder = Encoder(LATENT_DIM).to(device)
decoder = Decoder(LATENT_DIM).to(device)
unet = LatentUNet(LATENT_DIM, TIMESTEPS).to(device)

optimizer = optim.Adam(
    list(encoder.parameters()) +
    list(decoder.parameters()) +
    list(unet.parameters()),
    lr=LR
)

# ================= DIFFUSION SCHEDULER =================
betas = torch.linspace(1e-4, 0.02, TIMESTEPS).to(device)
alphas = 1. - betas
alpha_hat = torch.cumprod(alphas, dim=0)

# ================= KL LOSS =================
def kl_divergence(mu):
    return -0.5 * torch.sum(1 + 0 - mu.pow(2) - 1, dim=1).mean()

# ================= TRAIN =================
print("🚀 Training Started (224x224)...")

for epoch in range(EPOCHS):
    encoder.train()
    decoder.train()
    unet.train()

    pbar = tqdm(dataloader)

    for images, _ in pbar:
        images = images.to(device)

        with autocast(device_type=device.type):

            # ===== AUTOENCODER =====
            z = encoder(images)
            recon = decoder(z)

            recon_loss = F.mse_loss(recon, images)
            kl_loss = kl_divergence(z)
            ae_loss = recon_loss + 0.001 * kl_loss

            # ===== DIFFUSION =====
            t = torch.randint(0, TIMESTEPS, (z.size(0),), device=device).long()
            noise = torch.randn_like(z)

            a_hat = alpha_hat[t].unsqueeze(1)
            noisy_z = torch.sqrt(a_hat) * z + torch.sqrt(1 - a_hat) * noise

            pred_noise = unet(noisy_z, t)
            diffusion_loss = F.mse_loss(pred_noise, noise)

            # ===== TOTAL =====
            loss = ae_loss + diffusion_loss

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        pbar.set_description(f"Epoch {epoch+1} Loss: {loss.item():.4f}")

    # ===== SAVE =====
    if (epoch + 1) % 20 == 0:
        with torch.no_grad():
            sample_images, _ = next(iter(dataloader))
            z_sample = encoder(sample_images.to(device))
            samples = decoder(z_sample)

            save_image((samples + 1) / 2,
                       f"{SAVE_PATH}/sample_epoch{epoch+1}.png",
                       nrow=4)

            torch.save(encoder.state_dict(), f"{SAVE_PATH}/encoder_{epoch+1}.pth")
            torch.save(decoder.state_dict(), f"{SAVE_PATH}/decoder_{epoch+1}.pth")
            torch.save(unet.state_dict(),    f"{SAVE_PATH}/unet_{epoch+1}.pth")

            print(f"✅ Saved epoch {epoch+1}")

print("🎉 Training Completed!")

🚀 Training Started (224x224)...


Epoch 20 Loss: 0.8593: 100%|██████████| 183/183 [00:04<00:00, 39.25it/s]


✅ Saved epoch 20


Epoch 40 Loss: 0.7192: 100%|██████████| 183/183 [00:04<00:00, 38.21it/s]


✅ Saved epoch 40


...

✅ Saved epoch 460


Epoch 480 Loss: 0.5756: 100%|██████████| 183/183 [00:04<00:00, 38.18it/s]


✅ Saved epoch 480


Epoch 500 Loss: 0.6120: 100%|██████████| 183/183 [00:04<00:00, 38.78it/s]


✅ Saved epoch 500
🎉 Training Completed!


In [ ]:
# ✅ VANILLA LDM GENERATION CLASS-LEAFBLIGHT

import torch, os, shutil
from torchvision.utils import save_image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import torchvision.transforms as T
from itertools import cycle

# ================= CONFIG =================
LATENT_DIM = 1024
IMG_SIZE = 224
DATA_PATH = "/content/drive/MyDrive/BambooLeaf/LeafBlight"
OUTPUT_DIR = "/content/drive/MyDrive/BambooLeaf/LDMLB_224/LBGenerated"
ZIP_PATH = "/content/drive/MyDrive/BambooLeaf/LDMLB_224/LBGenerated.zip"
TOTAL_IMAGES = 5000

os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= DATASET =================
class SingleClassImageDataset(Dataset):
    def __init__(self, root_dir):
        self.paths = [os.path.join(root_dir, f) for f in os.listdir(root_dir)
                      if f.endswith((".png", ".jpg", ".jpeg"))]
        self.transform = T.Compose([
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img)

loader = DataLoader(SingleClassImageDataset(DATA_PATH),
                    batch_size=8, shuffle=True)

# ================= MODELS =================
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.Conv2d(512, 512, 4, 2, 1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, latent_dim)
        )
    def forward(self, x): return self.net(x)

class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 512 * 7 * 7)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh()
        )
    def forward(self, z):
        return self.net(self.fc(z).view(-1, 512, 7, 7))

# ================= LOAD =================
encoder = Encoder(LATENT_DIM).to(device)
decoder = Decoder(LATENT_DIM).to(device)

encoder.load_state_dict(torch.load("/content/drive/MyDrive/BambooLeaf/LDMLB_224/encoder_500.pth"))
decoder.load_state_dict(torch.load("/content/drive/MyDrive/BambooLeaf/LDMLB_224/decoder_500.pth"))

encoder.eval()
decoder.eval()

print("✅ Models loaded successfully (224x224)")

# ================= GENERATION =================
@torch.no_grad()
def generate():
    count = 0
    loop_loader = cycle(loader)

    while count < TOTAL_IMAGES:
        images = next(loop_loader).to(device)

        z = encoder(images)
        imgs = (decoder(z) + 1) / 2

        for j, img in enumerate(imgs):
            if count + j >= TOTAL_IMAGES:
                break
            save_image(img, f"{OUTPUT_DIR}/GEN_{count + j + 1:05}.png")

        count += len(imgs)

    print(f"✅ Generated {TOTAL_IMAGES} images")

# ================= ZIP FUNCTION =================
def create_zip():
    print("📦 Creating ZIP file...")
    shutil.make_archive(ZIP_PATH.replace(".zip", ""), 'zip', OUTPUT_DIR)
    print(f"✅ ZIP saved at: {ZIP_PATH}")

# ================= RUN =================
generate()
create_zip()

✅ Models loaded successfully (224x224)
✅ Generated 5000 images
📦 Creating ZIP file...
✅ ZIP saved at: /content/drive/MyDrive/BambooLeaf/LDMLB_224/LBGenerated.zip


In [ ]:
# ✅ VANILLA LDM TRAINING CLASS-DIEBACK

import torch, os
from torch import nn, optim
from torchvision import transforms as T
from torchvision.utils import save_image
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from torch.amp import autocast, GradScaler
import torch.nn.functional as F

# ================= CONFIG =================
IMG_SIZE = 224
LATENT_DIM = 1024   # you can reduce to 512 if needed
BATCH_SIZE = 8
EPOCHS = 500
LR = 2e-4
TIMESTEPS = 1000
DATA_PATH = "/content/drive/MyDrive/BambooLeaf/Dieback"

SAVE_PATH = "/content/drive/MyDrive/BambooLeaf/LDMDB_224"
os.makedirs(SAVE_PATH, exist_ok=True)

# ================= DEVICE =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = GradScaler()

# ================= DATASET =================
class SingleClassImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.paths = [os.path.join(root_dir, f) for f in os.listdir(root_dir)
                      if f.endswith((".png", ".jpg", ".jpeg"))]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, 0

# ================= TRANSFORM =================
transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

dataset = SingleClassImageDataset(DATA_PATH, transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# ================= MODELS =================

# ✅ ENCODER (224 → 7)
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.ReLU(),       # 224 → 112
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(), # 112 → 56
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(), # 56 → 28
            nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(), # 28 → 14
            nn.Conv2d(512, 512, 4, 2, 1), nn.ReLU(),                      # 14 → 7
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, latent_dim)
        )

    def forward(self, x):
        return self.net(x)

# ✅ DECODER (7 → 224)
class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 512 * 7 * 7)

        self.net = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(), # 7 → 14
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(), # 14 → 28
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(), # 28 → 56
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),   # 56 → 112
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh()                          # 112 → 224
        )

    def forward(self, z):
        x = self.fc(z).view(-1, 512, 7, 7)
        return self.net(x)

# ✅ LATENT UNET (Diffusion in latent space)
class LatentUNet(nn.Module):
    def __init__(self, latent_dim, timesteps):
        super().__init__()
        self.time_embed = nn.Embedding(timesteps, latent_dim)

        self.net = nn.Sequential(
            nn.Linear(latent_dim, 1024), nn.ReLU(),
            nn.Linear(1024, 1024), nn.ReLU(),
            nn.Linear(1024, latent_dim)
        )

    def forward(self, x, t):
        return self.net(x + self.time_embed(t))

# ================= INIT =================
encoder = Encoder(LATENT_DIM).to(device)
decoder = Decoder(LATENT_DIM).to(device)
unet = LatentUNet(LATENT_DIM, TIMESTEPS).to(device)

optimizer = optim.Adam(
    list(encoder.parameters()) +
    list(decoder.parameters()) +
    list(unet.parameters()),
    lr=LR
)

# ================= DIFFUSION SCHEDULER =================
betas = torch.linspace(1e-4, 0.02, TIMESTEPS).to(device)
alphas = 1. - betas
alpha_hat = torch.cumprod(alphas, dim=0)

# ================= KL LOSS =================
def kl_divergence(mu):
    return -0.5 * torch.sum(1 + 0 - mu.pow(2) - 1, dim=1).mean()

# ================= TRAIN =================
print("🚀 Training Started (224x224)...")

for epoch in range(EPOCHS):
    encoder.train()
    decoder.train()
    unet.train()

    pbar = tqdm(dataloader)

    for images, _ in pbar:
        images = images.to(device)

        with autocast(device_type=device.type):

            # ===== AUTOENCODER =====
            z = encoder(images)
            recon = decoder(z)

            recon_loss = F.mse_loss(recon, images)
            kl_loss = kl_divergence(z)
            ae_loss = recon_loss + 0.001 * kl_loss

            # ===== DIFFUSION =====
            t = torch.randint(0, TIMESTEPS, (z.size(0),), device=device).long()
            noise = torch.randn_like(z)

            a_hat = alpha_hat[t].unsqueeze(1)
            noisy_z = torch.sqrt(a_hat) * z + torch.sqrt(1 - a_hat) * noise

            pred_noise = unet(noisy_z, t)
            diffusion_loss = F.mse_loss(pred_noise, noise)

            # ===== TOTAL =====
            loss = ae_loss + diffusion_loss

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        pbar.set_description(f"Epoch {epoch+1} Loss: {loss.item():.4f}")

    # ===== SAVE =====
    if (epoch + 1) % 20 == 0:
        with torch.no_grad():
            sample_images, _ = next(iter(dataloader))
            z_sample = encoder(sample_images.to(device))
            samples = decoder(z_sample)

            save_image((samples + 1) / 2,
                       f"{SAVE_PATH}/sample_epoch{epoch+1}.png",
                       nrow=4)

            torch.save(encoder.state_dict(), f"{SAVE_PATH}/encoder_{epoch+1}.pth")
            torch.save(decoder.state_dict(), f"{SAVE_PATH}/decoder_{epoch+1}.pth")
            torch.save(unet.state_dict(),    f"{SAVE_PATH}/unet_{epoch+1}.pth")

            print(f"✅ Saved epoch {epoch+1}")

print("🎉 Training Completed!")

🚀 Training Started (224x224)...


Epoch 20 Loss: 1.0552: 100%|██████████| 157/157 [00:04<00:00, 38.06it/s]


✅ Saved epoch 20


Epoch 40 Loss: 0.7425: 100%|██████████| 157/157 [00:04<00:00, 38.09it/s]


✅ Saved epoch 40


...

✅ Saved epoch 460


Epoch 480 Loss: 0.5844: 100%|██████████| 157/157 [00:04<00:00, 38.31it/s]


✅ Saved epoch 480


Epoch 500 Loss: 0.5667: 100%|██████████| 157/157 [00:04<00:00, 38.45it/s]


✅ Saved epoch 500
🎉 Training Completed!


In [ ]:
# ✅ VANILLA LDM GENERATION CLASS-DIEBACK

import torch, os, shutil
from torchvision.utils import save_image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import torchvision.transforms as T
from itertools import cycle

# ================= CONFIG =================
LATENT_DIM = 1024
IMG_SIZE = 224
DATA_PATH = "/content/drive/MyDrive/BambooLeaf/Dieback"
OUTPUT_DIR = "/content/drive/MyDrive/BambooLeaf/LDMDB_224/DBGenerated"
ZIP_PATH = "/content/drive/MyDrive/BambooLeaf/LDMDB_224/LBGenerated.zip"
TOTAL_IMAGES = 5000

os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= DATASET =================
class SingleClassImageDataset(Dataset):
    def __init__(self, root_dir):
        self.paths = [os.path.join(root_dir, f) for f in os.listdir(root_dir)
                      if f.endswith((".png", ".jpg", ".jpeg"))]
        self.transform = T.Compose([
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img)

loader = DataLoader(SingleClassImageDataset(DATA_PATH),
                    batch_size=8, shuffle=True)

# ================= MODELS =================
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.Conv2d(512, 512, 4, 2, 1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, latent_dim)
        )
    def forward(self, x): return self.net(x)

class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 512 * 7 * 7)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh()
        )
    def forward(self, z):
        return self.net(self.fc(z).view(-1, 512, 7, 7))

# ================= LOAD =================
encoder = Encoder(LATENT_DIM).to(device)
decoder = Decoder(LATENT_DIM).to(device)

encoder.load_state_dict(torch.load("/content/drive/MyDrive/BambooLeaf/LDMDB_224/encoder_500.pth"))
decoder.load_state_dict(torch.load("/content/drive/MyDrive/BambooLeaf/LDMDB_224/decoder_500.pth"))

encoder.eval()
decoder.eval()

print("✅ Models loaded successfully (224x224)")

# ================= GENERATION =================
@torch.no_grad()
def generate():
    count = 0
    loop_loader = cycle(loader)

    while count < TOTAL_IMAGES:
        images = next(loop_loader).to(device)

        z = encoder(images)
        imgs = (decoder(z) + 1) / 2

        for j, img in enumerate(imgs):
            if count + j >= TOTAL_IMAGES:
                break
            save_image(img, f"{OUTPUT_DIR}/GEN_{count + j + 1:05}.png")

        count += len(imgs)

    print(f"✅ Generated {TOTAL_IMAGES} images")

# ================= ZIP FUNCTION =================
def create_zip():
    print("📦 Creating ZIP file...")
    shutil.make_archive(ZIP_PATH.replace(".zip", ""), 'zip', OUTPUT_DIR)
    print(f"✅ ZIP saved at: {ZIP_PATH}")

# ================= RUN =================
generate()
create_zip()

✅ Models loaded successfully (224x224)
✅ Generated 5000 images
📦 Creating ZIP file...
✅ ZIP saved at: /content/drive/MyDrive/BambooLeaf/LDMDB_224/LBGenerated.zip
